# Facial Keypoint Detection Entrenamiento CNN Baseline

Carga las imágenes preprocesadas desde los .npy, haciendo la ejecución muchisimo mas rápido


## 1. Imports

In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

try:
    import torch_directml
    device = torch_directml.device()
    print(f'Usando DirectML (AMD GPU): {device}')
except ImportError:
    device = torch.device('cpu')
    print('torch-directml no encontrado, usando CPU.')


## 2. Configuración

In [ ]:
# Ajustar la ruta donde estan todos los archivos
DATA_DIR    = r'F:\\DL_Project\\Data'
IMAGES_PATH    = os.path.join(DATA_DIR, 'preprocessed', 'images_50k.npy')
KEYPOINTS_PATH = os.path.join(DATA_DIR, 'preprocessed', 'keypoints_50k.npy')
MODEL_PATH     = os.path.join(DATA_DIR, 'best_baseline_cnn.pth')
LOSSES_PATH    = os.path.join(DATA_DIR, 'training_losses.pkl')

IMG_SIZE    = 96
BATCH_SIZE  = 64
NUM_EPOCHS  = 20
LR          = 1e-3
SEED        = 42

print(f'IMAGES_PATH:    {IMAGES_PATH}')
print(f'KEYPOINTS_PATH: {KEYPOINTS_PATH}')
print(f'NUM_EPOCHS:     {NUM_EPOCHS}')
print(f'BATCH_SIZE:     {BATCH_SIZE}')


IMAGES_PATH:    C:\Users\manic\OneDrive\Desktop\DL FinalProject\data\preprocessed\images_50k.npy
KEYPOINTS_PATH: C:\Users\manic\OneDrive\Desktop\DL FinalProject\data\preprocessed\keypoints_50k.npy
NUM_EPOCHS:     20
BATCH_SIZE:     64


## 3. Cargar datos y crear DataLoaders

In [ ]:
class FacialKeypointDataset(Dataset):
    def __init__(self, images, keypoints):

        self.images    = torch.tensor(images,    dtype=torch.float32)
        self.keypoints = torch.tensor(keypoints, dtype=torch.float32)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.keypoints[idx]


# Cargar arrays preprocesados
print('Cargando imágenes desde disco...')
images    = np.load(IMAGES_PATH)
keypoints = np.load(KEYPOINTS_PATH)
print(f'images:    {images.shape}   {images.nbytes / 1e9:.2f} GB')
print(f'keypoints: {keypoints.shape}   {keypoints.nbytes / 1e6:.1f} MB')

# Dataset y split 80/20
full_dataset = FacialKeypointDataset(images, keypoints)
torch.manual_seed(SEED)
train_size    = int(0.8 * len(full_dataset))
val_size      = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'\nTrain: {len(train_dataset):,} frames  ({len(train_loader)} batches)')
print(f'Val:   {len(val_dataset):,} frames  ({len(val_loader)} batches)')

imgs_b, kps_b = next(iter(train_loader))
print(f'\nBatch imágenes:  {imgs_b.shape}')
print(f'Batch keypoints: {kps_b.shape}')


## 4. Modelo — CNN Baseline

4 bloques Conv → ReLU → MaxPool + 3 capas FC.
Salida: 136 valores (x, y para los 68 keypoints) en [0, 1].


In [ ]:
class BaselineCNN(nn.Module):
    def __init__(self, num_keypoints=68):
        super().__init__()
        out_dim = num_keypoints * 2
        self.features = nn.Sequential(
            nn.Conv2d(3,   32, kernel_size=3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32,  64, kernel_size=3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(128,256, kernel_size=3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 6 * 6, 1024), nn.ReLU(inplace=True),
            nn.Linear(1024, 256),          nn.ReLU(inplace=True),
            nn.Linear(256, out_dim),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.regressor(self.features(x))


model = BaselineCNN().to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parámetros entrenables: {total_params:,}')

with torch.no_grad():
    dummy = torch.zeros(4, 3, IMG_SIZE, IMG_SIZE).to(device)
    out   = model(dummy)
    print(f'Input:  {dummy.shape}  →  Output: {out.shape}')


## 5. Loss y optimizador

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)
print('Loss:      MSELoss')
print('Optimizer: Adam  lr=1e-3')
print('Scheduler: ReduceLROnPlateau  factor=0.5  patience=3')


## 6. Entrenamiento

In [ ]:
best_val_loss = float('inf')
train_losses, val_losses = [], []

for epoch in range(1, NUM_EPOCHS + 1):

    #Train
    model.train()
    running_loss = 0.0
    for imgs, kps in train_loader:
        imgs, kps = imgs.to(device), kps.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), kps)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
    train_loss = running_loss / len(train_dataset)

    # Validation
    model.eval()
    running_val = 0.0
    with torch.no_grad():
        for imgs, kps in val_loader:
            imgs, kps = imgs.to(device), kps.to(device)
            running_val += criterion(model(imgs), kps).item() * imgs.size(0)
    val_loss = running_val / len(val_dataset)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), MODEL_PATH)
        saved_marker = '  ✅'
    else:
        saved_marker = ''

    print(f'Epoch {epoch:2d}/{NUM_EPOCHS}  '
          f'train: {train_loss:.6f}  '
          f'val: {val_loss:.6f}  '
          f'lr: {current_lr:.2e}'
          f'{saved_marker}')

print(f'\nMejor val_loss: {best_val_loss:.6f}')
print(f'Modelo guardado en: {MODEL_PATH}')

with open(LOSSES_PATH, 'wb') as f:
    pickle.dump({'train': train_losses, 'val': val_losses}, f)
print(f'Losses guardadas en: {LOSSES_PATH}')


## 7. Curva de pérdida

In [ ]:
epochs = range(1, len(train_losses) + 1)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(epochs, train_losses, label='Train loss',      color='steelblue')
ax.plot(epochs, val_losses,   label='Validation loss', color='tomato')
ax.set_xlabel('Época'); ax.set_ylabel('MSE Loss')
ax.set_title('Curva de pérdida — CNN Baseline')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
